# Notebook 04: Limpieza, Transformación y Normalización Canónica (Capa Silver)
**Proyecto**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Fase PDCO**: DEVELOPMENT $\rightarrow$ CONTROL | **Capa**: Bronze $\rightarrow$ Silver  
**Estándar**: DAMA-DMBOK 2 / Clean Architecture / SWEBOK v4  

---
### Objetivos de la Capa Silver
1. **Sanitización Tipográfica**: Eliminación de acentos diacríticos (Unicode NFKD), colapso de espacios dobles y unificación a mayúsculas.
2. **Imputación Controlada**: Tratamiento determinístico de valores nulos (`SIN_VARIEDAD_ESPECIFICADA`).
3. **Tipado Estricto**: Coerción de fechas a `datetime64`, identificadores a `string`, y métricas a `float64` / `int64`.
4. **Deduplicación Temporal y Llaves Compuestas**: Consolidación sobre llaves primarias de negocio.
5. **Persistencia Parquet Snappy**: Almacenamiento optimizado y cálculo de huella criptográfica SHA-256 en `data/CLEANED/`.

In [1]:
# Celda 1: Setup e Importaciones de Módulos de Limpieza
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import settings
from src.utils.logger import get_logger
from src.ingestion.landing_manager import LandingManager
from src.cleaning.transformers import (
    sanitize_text,
    cast_datatypes,
    handle_missing_values,
    deduplicate_dataset,
)
from src.cleaning.cleaner_pipeline import CleanerPipeline

logger = get_logger("notebooks.04_cleaning")
landing = LandingManager()
cleaner = CleanerPipeline()
print(f"CleanerPipeline inicializado. Directorio Silver: {cleaner.cleaned_dir}")

CleanerPipeline inicializado. Directorio Silver: C:\Users\ADAN\OneDrive\Documentos\App_agro\AgroStatsApp\data\CLEANED


### 1. Limpieza y Normalización de Evaluaciones Agropecuarias (EVA)

In [2]:
# Celda 2: Carga y saneamiento de datos agrícolas EVA
eva_path = settings.RAW_DIR / "upra" / "evaluaciones_agropecuarias_eva.parquet"
if eva_path.exists():
    df_eva_raw = pd.read_parquet(eva_path)
    df_eva_silver = cleaner.clean_eva_dataset(df_eva_raw)
    print(f"EVA Raw Filas: {len(df_eva_raw)} -> EVA Silver Filas: {len(df_eva_silver)}")
    display(df_eva_silver.head(3))
else:
    print(f"Aviso: Dataset {eva_path.name} no encontrado en Bronze. Se generará muestra de validación.")
    df_eva_mock = pd.DataFrame({
        'c_d_dep': ['05'], 'c_d_mun': ['05001'], 'departamento': ['Antioquia'],
        'municipio': ['Medellín'], 'cultivo': ['Café'], 'desagregaci_n_regional_y': [None],
        'a_o': ['2023'], 'rea_sembrada_ha': ['150.0'], 'rea_cosechada_ha': ['145.0'],
        'producci_n_t': ['320.0'], 'rendimiento_t_ha': ['2.21']
    })
    df_eva_silver = cleaner.clean_eva_dataset(df_eva_mock)
    display(df_eva_silver)

Aviso: Dataset evaluaciones_agropecuarias_eva.parquet no encontrado en Bronze. Se generará muestra de validación.


,cod_departamento,cod_municipio,departamento,municipio,cultivo,variedad,anio,area_sembrada_ha,area_cosechada_ha,produccion_ton,rendimiento_ton_ha
0,05,05001,ANTIOQUIA,MEDELLIN,CAFE,SIN_VARIEDAD_ESPECIFICADA,2023,150.0,145.0,320.0,2.21


### 2. Saneamiento de Series Meteorológicas de IDEAM

In [3]:
# Celda 3: Normalización y tipado de variables meteorológicas (IDEAM)
ideam_path = settings.RAW_DIR / "ideam" / "precipitacion_pluviometrica.parquet"
if ideam_path.exists():
    df_ideam_raw = pd.read_parquet(ideam_path)
    df_ideam_silver = cleaner.clean_ideam_dataset(df_ideam_raw)
    print(f"IDEAM Raw Filas: {len(df_ideam_raw)} -> IDEAM Silver Filas: {len(df_ideam_silver)}")
    display(df_ideam_silver.head(3))
else:
    print(f"Aviso: Archivo {ideam_path.name} no encontrado en Bronze. Verificando pipeline con mock.")
    df_ideam_mock = pd.DataFrame({
        'codigoestacion': ['21206990'], 'nombreestacion': ['El Salado'],
        'fechaobservacion': ['2023-05-15 07:00:00'], 'valorobservado': ['12.5'],
        'descripcionsensor': ['Precipitación Pluviómetro']
    })
    df_ideam_silver = cleaner.clean_ideam_dataset(df_ideam_mock)
    display(df_ideam_silver)

Aviso: Archivo precipitacion_pluviometrica.parquet no encontrado en Bronze. Verificando pipeline con mock.


,codigo_estacion,nombre_estacion,fecha_observacion,valor_observado,sensor_descripcion
0,21206990,El Salado,2023-05-15 07:00:00,12.5,PRECIPITACION PLUVIOMETRO


### 3. Normalización de Series de Precios e Insumos SIPSA / DANE

In [4]:
# Celda 4: Saneamiento de insumos y precios mayoristas SIPSA
sipsa_path = settings.RAW_DIR / "dane" / "insumos_fertilizantes_sipsa.parquet"
if sipsa_path.exists():
    df_sipsa_raw = pd.read_parquet(sipsa_path)
    df_sipsa_silver = cleaner.clean_sipsa_dataset(df_sipsa_raw)
    print(f"SIPSA Raw Filas: {len(df_sipsa_raw)} -> SIPSA Silver Filas: {len(df_sipsa_silver)}")
    display(df_sipsa_silver.head(3))
else:
    print(f"Aviso: Archivo {sipsa_path.name} no encontrado en Bronze. Validando con muestra.")
    df_sipsa_mock = pd.DataFrame({
        'cod_municipio': ['11001'], 'municipio': ['Bogotá, D.C.'],
        'articulo': ['Urea 46%'], 'preciopromedio': ['145000.0'],
        'anio': ['2023'], 'mes': ['6']
    })
    df_sipsa_silver = cleaner.clean_sipsa_dataset(df_sipsa_mock)
    display(df_sipsa_silver)

Aviso: Archivo insumos_fertilizantes_sipsa.parquet no encontrado en Bronze. Validando con muestra.


,cod_municipio,municipio,articulo,precio_promedio,anio,mes
0,11001,"BOGOTA, D.C.",UREA 46%,145000.0,2023,6


### 4. Orquestación y Persistencia Masiva hacia la Capa Silver (`data/CLEANED/`)

In [5]:
# Celda 5: Procesamiento sistemático de datasets de Bronze a Silver
silver_catalog = []
raw_files = list(settings.RAW_DIR.rglob("*.parquet"))
print(f"Archivos Parquet identificados en Bronze: {len(raw_files)}")

for fpath in raw_files:
    category = fpath.parent.name
    stem = fpath.stem
    df_raw = pd.read_parquet(fpath)
    
    cleaner_type = "generic"
    if "eva" in stem:
        cleaner_type = "eva"
    elif "exportaciones" in stem or "bioinsumos" in stem:
        cleaner_type = "trade"
    elif "ideam" in category or any(w in stem for w in ["precipitacion", "temperatura", "radiacion"]):
        cleaner_type = "ideam"
    elif "sipsa" in stem or category == "dane":
        cleaner_type = "sipsa"
        
    summary = cleaner.process_and_save(df_raw, dataset_name=f"{stem}_silver", cleaner_type=cleaner_type)
    silver_catalog.append(summary)
    print(f"[SILVER] {stem} -> Filas: {summary['final_rows']} | SHA-256: {summary['sha256'][:12]}...")

Archivos Parquet identificados en Bronze: 11


2026-10-07T20:00:27-0500 | WARNING  | agrostats.cleaning.transformers:deduplicate_dataset:127 | deduplicate_dataset invocado sin claves válidas presentes en el DataFrame
2026-10-07T20:00:27-0500 | INFO     | agrostats.cleaning.cleaner_pipeline:process_and_save:193 | Capa Silver generada para sipsa_alimentos_balanceados_silver: 68 filas (reducción: 0 duplicados)


[SILVER] sipsa_alimentos_balanceados -> Filas: 68 | SHA-256: d2a6efbacf5a...


2026-10-07T20:00:27-0500 | WARNING  | agrostats.cleaning.transformers:deduplicate_dataset:127 | deduplicate_dataset invocado sin claves válidas presentes en el DataFrame
2026-10-07T20:00:27-0500 | INFO     | agrostats.cleaning.cleaner_pipeline:process_and_save:193 | Capa Silver generada para sipsa_insumos_precios_silver: 93 filas (reducción: 0 duplicados)
2026-10-07T20:00:27-0500 | WARNING  | agrostats.cleaning.transformers:deduplicate_dataset:127 | deduplicate_dataset invocado sin claves válidas presentes en el DataFrame
2026-10-07T20:00:27-0500 | INFO     | agrostats.cleaning.cleaner_pipeline:process_and_save:193 | Capa Silver generada para normales_spi_silver: 5000 filas (reducción: 0 duplicados)


[SILVER] sipsa_insumos_precios -> Filas: 93 | SHA-256: e66e97a050fa...
[SILVER] normales_spi -> Filas: 5000 | SHA-256: 648c2915658b...


...

[SILVER] temperatura_maxima -> Filas: 5000 | SHA-256: 24a5c65dc440...


2026-10-07T20:00:27-0500 | INFO     | agrostats.cleaning.cleaner_pipeline:process_and_save:193 | Capa Silver generada para temperatura_media_silver: 5000 filas (reducción: 0 duplicados)
2026-10-07T20:00:27-0500 | INFO     | agrostats.cleaning.cleaner_pipeline:process_and_save:193 | Capa Silver generada para temperatura_minima_silver: 5000 filas (reducción: 0 duplicados)


[SILVER] temperatura_media -> Filas: 5000 | SHA-256: 9129d005048e...
[SILVER] temperatura_minima -> Filas: 5000 | SHA-256: dd60342d36f3...


2026-10-07T20:00:27-0500 | INFO     | agrostats.cleaning.transformers:deduplicate_dataset:139 | Deduplicación eliminó 88 registros duplicados sobre claves ['cod_municipio', 'anio', 'cultivo', 'variedad']
2026-10-07T20:00:28-0500 | INFO     | agrostats.cleaning.cleaner_pipeline:process_and_save:193 | Capa Silver generada para upra_eva_historico_silver: 4912 filas (reducción: 88 duplicados)
2026-10-07T20:00:28-0500 | INFO     | agrostats.cleaning.transformers:deduplicate_dataset:139 | Deduplicación eliminó 4999 registros duplicados sobre claves ['anio']
2026-10-07T20:00:28-0500 | INFO     | agrostats.cleaning.cleaner_pipeline:process_and_save:193 | Capa Silver generada para upra_exportaciones_bioinsumos_silver: 1 filas (reducción: 4999 duplicados)


[SILVER] upra_eva_historico -> Filas: 4912 | SHA-256: 4b232b7ef8b8...
[SILVER] upra_exportaciones_bioinsumos -> Filas: 1 | SHA-256: 8380bbc106d1...


### 5. Resumen de Calidad y Trazabilidad Criptográfica de la Capa Silver

In [6]:
# Celda 6: Resumen consolidado de la capa Silver
if silver_catalog:
    df_summary = pd.DataFrame(silver_catalog)
    display(df_summary[["dataset_name", "initial_rows", "final_rows", "removed_duplicates", "file_size_bytes", "sha256"]])
else:
    print("No se procesaron archivos en este entorno de demostración.")

,dataset_name,initial_rows,final_rows,removed_duplicates,file_size_bytes,sha256
0,sipsa_alimentos_balanceados_silver,68,68,0,6941,d2a6efbacf5a9c5fcf27f3cfd3f0a11b09f42c310a5b95...
1,sipsa_insumos_precios_silver,93,93,0,68514,e66e97a050faf67b58ae31663a8b5e114a4ff78c481cb9...
2,normales_spi_silver,5000,5000,0,296640,648c2915658b9a79d01fc70105f5f9bae48fbd361644ef...
3,pluviometria_estaciones_silver,5000,5000,0,118116,a48919d98dde943d8d5449e889fb977778f5f81a8caa86...
4,radiacion_solar_silver,169,169,0,33900,0b1e92b50ec315061dce0d0f5d5818680f88a7158064cd...
5,telemetria_sensores_silver,5000,5000,0,109875,222eb915ae2145e1509963aa0eca7e3bc36e3e49fabe1a...
6,temperatura_maxima_silver,5000,5000,0,109476,24a5c65dc440c73e81aa8eb9c7323f6f540ec74a675dc5...
7,temperatura_media_silver,5000,5000,0,68834,9129d005048edbfcb17966ddc25a732bfe8429f8e69c0a...
8,temperatura_minima_silver,5000,5000,0,110734,dd60342d36f38a0abbcea3d4a2b71722c0c831bda79c43...
9,upra_eva_historico_silver,5000,4912,88,68772,4b232b7ef8b8366b151e4d8c86dffff86d657db3ee2adf...
